# 🏟️ Ball Detection avec YOLOv8 — Vidéo tactical complète

**Objectif** : Détecter le ballon sur les 199319 frames de la vidéo tactical  
pour compléter les positions manquantes du tracker ATD.

---
### 📋 Ce dont tu as besoin dans ton Google Drive
```
Mon Drive/
└── mtah2027/
    ├── BAR-ATH_tactical.mp4              ← vidéo complète (~5,2 Go)
    └── BAR-ATH_tactical_homography.csv   ← pour convertir pixels → mètres
```

## Étape 1 — Setup : GPU + librairies

In [ ]:
# Vérifier que le GPU T4 est bien activé
# (Runtime > Change runtime type > T4 GPU si ce n'est pas fait)
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ Pas de GPU détecté — active le GPU dans Runtime > Change runtime type')

In [ ]:
# Installer YOLOv8 (ultralytics)
!pip install ultralytics --quiet
print('✅ ultralytics installé')

## Étape 2 — Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive monté')

In [ ]:
import os

# ⚙️ Dossier contenant les fichiers tactical dans Google Drive
DRIVE_DIR       = '/content/drive/MyDrive/mtah2027'
VIDEO_PATH      = os.path.join(DRIVE_DIR, 'BAR-ATH_tactical.mp4')
HOMOGRAPHY_PATH = os.path.join(DRIVE_DIR, 'BAR-ATH_tactical_homography.csv')
OUTPUT_CSV      = os.path.join(DRIVE_DIR, 'yolo_ball_full.csv')
CHECKPOINT_CSV  = os.path.join(DRIVE_DIR, 'yolo_ball_full_checkpoint.csv')

# Vérification
for path, name in [(VIDEO_PATH, 'Vidéo tactical complète'), (HOMOGRAPHY_PATH, 'Homography')]:
    status = '✅' if os.path.exists(path) else '❌ MANQUANT'
    print(f'{status}  {name}: {path}')

## Étape 3 — Constantes du match

In [ ]:
# Toute la vidéo tactical, indexée comme la soumission
START_FRAME      = 0
N_FRAMES         = 199319
FPS              = 25
PITCH_LENGTH_M   = 105.3   # mètres
PITCH_WIDTH_M    = 68.0    # mètres

# Seuil de confiance YOLO : ne garder que les détections fiables
CONF_THRESHOLD   = 0.25

print(f'Vidéo : frames {START_FRAME} → {START_FRAME + N_FRAMES - 1}')
print(f'Soit {N_FRAMES} frames à analyser (~{N_FRAMES/FPS/60:.1f} minutes)')

## Étape 4 — Charger la Homographie

In [ ]:
import numpy as np
import pandas as pd

def load_homography(csv_path):
    """Charge la homographie. Retourne un array (N, 3, 3) et le masque valid."""
    df = pd.read_csv(csv_path)
    H_cols = ['h00','h01','h02','h10','h11','h12','h20','h21','h22']
    H_flat = df[H_cols].values                      # shape (N, 9)
    H      = H_flat.reshape(-1, 3, 3)               # shape (N, 3, 3)
    valid  = df['valid'].values.astype(bool)         # shape (N,)
    frames = df['frame'].values                      # video frame index
    return H, valid, frames

H_all, valid_all, frame_ids_all = load_homography(HOMOGRAPHY_PATH)
print(f'✅ Homographie chargée : {len(H_all)} matrices')
print(f'   Valides : {valid_all.sum()} / {len(valid_all)} ({100*valid_all.mean():.1f}%)')

In [ ]:
def pixel_to_pitch(H_matrix, px, py):
    """
    Convertit un point pixel (px, py) en coordonnées terrain en mètres
    via la matrice homographie inverse.
    H_matrix mappe terrain_norm [0,1] → pixel.
    On inverse pour aller pixel → terrain_norm → mètres.
    """
    try:
        H_inv = np.linalg.inv(H_matrix)
    except np.linalg.LinAlgError:
        return np.nan, np.nan

    point = np.array([px, py, 1.0])
    result = H_inv @ point
    if abs(result[2]) < 1e-10:
        return np.nan, np.nan

    x_norm = result[0] / result[2]   # coordonnée normalisée [0, 1]
    y_norm = result[1] / result[2]
    return x_norm * PITCH_LENGTH_M, y_norm * PITCH_WIDTH_M


def get_homography_for_frame(video_frame, H_all, valid_all, frame_ids_all):
    """Récupère la matrice H pour une frame vidéo donnée."""
    idx = np.searchsorted(frame_ids_all, video_frame)
    if idx >= len(frame_ids_all) or frame_ids_all[idx] != video_frame:
        return None, False
    return H_all[idx], valid_all[idx]

print('✅ Fonctions homographie prêtes')

## Étape 5 — Détection YOLO frame par frame

In [ ]:
from ultralytics import YOLO
import cv2

# Charger YOLOv8 pré-entraîné sur COCO
# 'sports ball' est la classe 32 dans COCO
model = YOLO('yolov8n.pt')   # 'n' = nano (le plus rapide, suffisant pour un ballon)
BALL_CLASS_ID = 32            # classe COCO 'sports ball'

print('✅ YOLOv8 nano chargé')
print(f'   Classe cible : {model.names[BALL_CLASS_ID]} (id={BALL_CLASS_ID})')

In [ ]:
results_rows = []
start_frame = START_FRAME
if os.path.exists(CHECKPOINT_CSV):
    checkpoint = pd.read_csv(CHECKPOINT_CSV)
    results_rows = checkpoint.to_dict('records')
    if results_rows:
        start_frame = int(results_rows[-1]['video_frame']) + 1
    print(f'Reprise à la frame {start_frame}')

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f'❌ Impossible d\'ouvrir la vidéo : {VIDEO_PATH}')

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if total_frames < N_FRAMES:
    raise IOError(f'La vidéo contient {total_frames} frames, {N_FRAMES} attendues')
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
print(f'Vidéo ouverte : {total_frames} frames')
print(f'Traitement en cours... (sauvegarde auto toutes les 1000 frames)\n')

frame_idx = start_frame

while frame_idx < N_FRAMES:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    # Frame réelle dans la vidéo complète
    video_frame = frame_idx

    # ── Détection YOLO ───────────────────────────────────────────────────────
    preds = model(
        frame_bgr,
        classes=[BALL_CLASS_ID],   # cherche uniquement 'sports ball'
        conf=CONF_THRESHOLD,
        verbose=False
    )

    ball_px, ball_py, confidence = np.nan, np.nan, np.nan
    x_m, y_m = np.nan, np.nan

    if preds and len(preds[0].boxes) > 0:
        # Prendre la détection avec la confiance la plus haute
        boxes     = preds[0].boxes
        best_idx  = int(boxes.conf.argmax())
        confidence = float(boxes.conf[best_idx])

        # Centre de la bounding box en pixels
        xyxy   = boxes.xyxy[best_idx].cpu().numpy()
        ball_px = (xyxy[0] + xyxy[2]) / 2
        ball_py = (xyxy[1] + xyxy[3]) / 2

        # ── Conversion pixel → mètres via homographie ────────────────────────
        H_mat, is_valid = get_homography_for_frame(
            video_frame, H_all, valid_all, frame_ids_all)

        if is_valid and H_mat is not None:
            x_m, y_m = pixel_to_pitch(H_mat, ball_px, ball_py)

            # Sanity check : le ballon doit être sur le terrain
            if not (0 <= x_m <= PITCH_LENGTH_M and 0 <= y_m <= PITCH_WIDTH_M):
                x_m, y_m = np.nan, np.nan  # hors terrain → ignorer

    results_rows.append({
        'video_frame': video_frame,
        'clip_frame':  frame_idx - START_FRAME,
        'ball_px':     round(ball_px, 1) if not np.isnan(ball_px) else None,
        'ball_py':     round(ball_py, 1) if not np.isnan(ball_py) else None,
        'confidence':  round(confidence, 3) if not np.isnan(confidence) else None,
        'x_m':         round(x_m, 3) if not np.isnan(x_m) else None,
        'y_m':         round(y_m, 3) if not np.isnan(y_m) else None,
    })

    # Sauvegarde intermédiaire toutes les 1000 frames (sécurité anti-timeout)
    if (frame_idx + 1) % 1000 == 0:
        pd.DataFrame(results_rows).to_csv(OUTPUT_CSV, index=False)
        detected = sum(1 for r in results_rows if r['x_m'] is not None)
        pd.DataFrame(results_rows).to_csv(CHECKPOINT_CSV, index=False)
        print(f'  Frame {frame_idx+1:6d}/{N_FRAMES} '
              f'| ballon détecté sur {detected} frames '
              f'({100*detected/(frame_idx+1):.1f}%) '
              f'| sauvegardé ✅')

    frame_idx += 1

cap.release()

# Sauvegarde finale
df_results = pd.DataFrame(results_rows)
df_results.to_csv(OUTPUT_CSV, index=False)
print(f'\n✅ Terminé ! {len(df_results)} frames traitées')
print(f'   Résultats sauvegardés : {OUTPUT_CSV}')

## Étape 6 — Analyse des résultats

In [ ]:
df = pd.read_csv(OUTPUT_CSV)

detected   = df['x_m'].notna()
n_detected = detected.sum()
n_total    = len(df)

print('=== Résumé de la détection YOLO ===')
print(f'Frames totales    : {n_total}')
print(f'Ballon détecté    : {n_detected} ({100*n_detected/n_total:.1f}%)')
print(f'Ballon non vu     : {n_total - n_detected} ({100*(n_total-n_detected)/n_total:.1f}%)')
print(f'Confiance moyenne : {df["confidence"].mean():.3f}')
print()
print('Distribution des positions détectées (mètres):')
print(df[['x_m','y_m']].describe().round(2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Heatmap des positions du ballon ──────────────────────────────────────────
ax = axes[0]
df_det = df[detected]
ax.hist2d(df_det['x_m'], df_det['y_m'], bins=40, cmap='hot')
ax.set_xlim(0, PITCH_LENGTH_M)
ax.set_ylim(0, PITCH_WIDTH_M)
ax.set_title('Heatmap des positions du ballon détectées')
ax.set_xlabel('Longueur terrain (m)')
ax.set_ylabel('Largeur terrain (m)')
rect = patches.Rectangle((0,0), PITCH_LENGTH_M, PITCH_WIDTH_M,
                           linewidth=2, edgecolor='white', facecolor='none')
ax.add_patch(rect)

# ── Confiance au fil du temps ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(df['clip_frame'], df['confidence'], alpha=0.5, linewidth=0.5, color='cyan')
ax2.axhline(CONF_THRESHOLD, color='red', linestyle='--', label=f'Seuil ({CONF_THRESHOLD})')
ax2.set_title('Confiance YOLO par frame')
ax2.set_xlabel('Frame (clip)')
ax2.set_ylabel('Confiance')
ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'yolo_analysis.png'), dpi=150)
plt.show()
print('✅ Graphique sauvegardé dans Drive')

## Étape 7 — Extraire les positions ATD complètes

Cette cellule crée automatiquement le fichier ATD attendu par la fusion. Elle conserve une ligne par frame et convertit les coordonnées normalisées en mètres.

In [ ]:
ATD_PATH = os.path.join(DRIVE_DIR, 'BAR-ATH_tactical_atd.txt')
ATD_CSV_PATH = os.path.join(DRIVE_DIR, 'atd_ball_full.csv')
N_FRAMES = 199319
PITCH_LENGTH_M = 105.3
PITCH_WIDTH_M = 68.0

atd_x = np.full(N_FRAMES, np.nan, dtype=float)
atd_y = np.full(N_FRAMES, np.nan, dtype=float)

with open(ATD_PATH) as handle:
    for line_number, line in enumerate(handle, start=1):
        parts = line.rstrip('\r\n').split(':')
        if len(parts) != 3:
            raise ValueError(f'Format ATD invalide à la ligne {line_number}')
        frame = int(parts[0]) - 1
        if not 0 <= frame < N_FRAMES:
            raise ValueError(f'Frame ATD hors limites: {frame}')
        values = parts[2].split(',')
        if len(values) >= 2 and not values[0].startswith('N'):
            atd_x[frame] = float(values[0]) * PITCH_LENGTH_M
            atd_y[frame] = float(values[1]) * PITCH_WIDTH_M

df_atd = pd.DataFrame({
    'video_frame': np.arange(N_FRAMES, dtype=np.int64),
    'atd_x_m': atd_x,
    'atd_y_m': atd_y,
})
df_atd.to_csv(ATD_CSV_PATH, index=False)
print(f'✅ ATD créé : {ATD_CSV_PATH}')
print(f'Frames : {len(df_atd)} | positions valides : {df_atd.atd_x_m.notna().sum()}')

## Étape 7 — Fusion avec l'ATD (combler les NaN)

On utilise YOLO **uniquement** pour combler les frames où l'ATD n'a pas de ballon.

In [ ]:
# Charger les positions ATD de toute la vidéo tactical
ATD_CSV_PATH = os.path.join(DRIVE_DIR, 'atd_ball_full.csv')  # colonnes: video_frame, atd_x_m, atd_y_m

if os.path.exists(ATD_CSV_PATH):
    df_atd = pd.read_csv(ATD_CSV_PATH)

    df_merged = df_atd.merge(df[['video_frame','x_m','y_m','confidence']],
                              on='video_frame', how='left')

    # Règle de fusion :
    # 1. ATD disponible → utiliser ATD  (source de référence)
    # 2. ATD manquant + YOLO confiance >= 0.40 → utiliser YOLO
    # 3. Les deux manquants → NaN (gap long, laisser à l'interpolation mathématique)
    df_merged['final_x_m'] = df_merged['atd_x_m']
    df_merged['final_y_m'] = df_merged['atd_y_m']
    df_merged['source']    = np.where(df_merged['atd_x_m'].notna(), 'ATD', 'unresolved')

    yolo_fills = df_merged['atd_x_m'].isna() & (df_merged['confidence'] >= 0.40)
    df_merged.loc[yolo_fills, 'final_x_m'] = df_merged.loc[yolo_fills, 'x_m']
    df_merged.loc[yolo_fills, 'final_y_m'] = df_merged.loc[yolo_fills, 'y_m']
    df_merged.loc[yolo_fills, 'source']    = 'YOLO'

    print(f'ATD frames valides  : {df_merged["atd_x_m"].notna().sum()}')
    print(f'Frames remplies YOLO: {yolo_fills.sum()}')
    print(f'Frames encore NaN   : {df_merged["final_x_m"].isna().sum()}')
    print(f'  → Ces frames seront gérées par l\'interpolation mathématique (ball_interpolation.py)')

    output_merged = os.path.join(DRIVE_DIR, 'ball_positions_merged_full.csv')
    df_merged[['video_frame','final_x_m','final_y_m','source']].to_csv(
        output_merged, index=False)
    print(f'\n✅ Fichier fusionné : {output_merged}')
else:
    print('⚠️  Fichier ATD non trouvé.')
    print('   Génère d’abord atd_ball_full.csv à partir de l’ATD complet')
    print(f'   Le CSV YOLO seul est disponible : {OUTPUT_CSV}')

---
## ✅ Résumé du pipeline complet

```
BAR-ATH_tactical.mp4  (vidéo tactical complète)
        │
        ▼
   YOLOv8 nano  (cette frame)
        │
        ▼
   pixel (px, py)  +  homography.csv  →  mètres (x_m, y_m)
        │
        ▼
   Fusion avec ATD :
   ┌─────────────────────────────────────────────┐
   │  ATD disponible          → garder ATD       │
   │  ATD manquant + conf≥0.4 → utiliser YOLO    │
   │  Toujours NaN            → interpolation    │
   └─────────────────────────────────────────────┘
        │
        ▼
   ball_positions_merged_full.csv
        │
        ▼
   → Équipe : logique de possession
```

**Fichiers produits dans ton Drive :**
- `yolo_ball_full.csv` — détections brutes YOLO (frame, pixels, confiance, mètres)
- `yolo_analysis.png` — heatmap + courbe de confiance
- `ball_positions_merged_full.csv` — positions finales fusionnées ATD+YOLO